Good. Now we connect the camera’s **ray time** to the sphere.

This is the second half of motion blur.

Before:

```text id="2aw5df"
Camera sends time
Sphere ignores time
→ no motion blur
```

After:

```text id="utgh0c"
Camera sends time
Sphere reads time
Sphere moves
→ motion blur works
```

---

# Core idea

Static sphere:

Center is fixed:

$$
C
$$

Moving sphere:

Center changes with time:

$$
C(t)=C_1+t(C_2-C_1)
$$

This is linear interpolation.

---

# Why store center as a ray?

Book does:

$$
center(t)=center_1+t(center_2-center_1)
$$

That is exactly ray form:

$$
R(t)=A+tB
$$

where:

$$
A=center_1
$$

$$
B=center_2-center_1
$$

So the sphere center itself becomes a ray.

Very elegant.

---

# Learning Table

| Step            | Formula               | Purpose               |
| --------------- | --------------------- | --------------------- |
| Static center   | $C$                   | fixed object          |
| Moving center   | $C(t)=C_1+t(C_2-C_1)$ | animate object        |
| Current center  | $C(r.time())$         | synchronize with ray  |
| Sphere equation | $|P-C(t)|^2=r^2$      | moving intersection   |
| Normal          | $\frac{P-C(t)}{r}$    | correct moving normal |

---

# Full Updated Sphere Class (Motion Blur Ready)

```python id="y8ay8f"
import math

from util.week1.vec3 import dot
from util.week1.hittable import Hittable
from util.week1.ray import Ray


class Sphere(Hittable):

    def __init__(
        self,
        center1,
        radius,
        material,
        center2=None
    ):
        """
        ============================================
        SPHERE CONSTRUCTOR
        ============================================

        Supports:

        1. Static sphere
        2. Moving sphere

        --------------------------------------------
        STATIC:

        Sphere(center, radius, material)

        center(t) = center

        --------------------------------------------
        MOVING:

        Sphere(center1, radius, material, center2)

        center(t)=center1+t(center2-center1)

        We store motion as a ray.
        """

        self.radius = max(0.0, radius)
        self.mat = material

        # ============================================
        # STATIC SPHERE
        # OLD behavior
        # ============================================
        if center2 is None:

            # CHANGED:
            # store zero-velocity ray
            self.center = Ray(
                center1,
                center1 * 0
            )

        # ============================================
        # MOVING SPHERE
        # NEW behavior
        # ============================================
        else:

            # center(t)=center1+t(center2-center1)

            self.center = Ray(
                center1,
                center2 - center1
            )

    # ==================================================
    # HIT TEST
    # ==================================================
    def hit(self, r, ray_t, rec):

        """
        ============================================
        STEP 1:
        Compute current center at ray time
        ============================================

        Formula:

        C(t)=C1+t(C2-C1)

        CHANGED:
        Before:
            fixed center

        Now:
            moving center
        """

        current_center = self.center.at(r.time())

        """
        Sphere equation:

        (P-C(t))·(P-C(t)) = r²

        Ray equation:

        P = O+tD
        """

        # ============================================
        # Vector from ray origin to moving center
        # ============================================
        oc = current_center - r.origin()

        # a = D·D
        a = dot(
            r.direction(),
            r.direction()
        )

        # h = D·oc
        h = dot(
            r.direction(),
            oc
        )

        # c = oc·oc - r²
        c = dot(
            oc,
            oc
        ) - self.radius * self.radius

        # ============================================
        # Discriminant
        #
        # h²-ac
        # ============================================
        discriminant = h * h - a * c

        if discriminant < 0:
            return False

        sqrtd = math.sqrt(discriminant)

        # ============================================
        # Find nearest root
        # ============================================
        root = (h - sqrtd) / a

        if not ray_t.surrounds(root):

            root = (h + sqrtd) / a

            if not ray_t.surrounds(root):
                return False

        # ============================================
        # Store hit
        # ============================================
        rec.t = root

        # P=O+tD
        rec.p = r.at(root)

        """
        ============================================
        CHANGED:
        normal uses current_center

        Before:
            rec.p - self.center

        Wrong for moving sphere

        Now:
            rec.p - current_center
        ============================================
        """

        outward_normal = (
            rec.p - current_center
        ) / self.radius

        rec.set_face_normal(
            r,
            outward_normal
        )

        # material
        rec.mat = self.mat

        return True
```

---

# What changed?

| Old                    | New                                    |
| ---------------------- | -------------------------------------- |
| `self.center = center` | `self.center = Ray(center1, velocity)` |
| fixed sphere           | moving sphere                          |
| `self.center` in hit   | `current_center = center.at(r.time())` |
| static normal          | moving normal                          |

---

# Usage

Static:

```python id="ut2uav"
sphere = Sphere(
    Vec3(0,0,-1),
    0.5,
    material
)
```

---

Moving:

```python id="zuz1kt"
sphere = Sphere(
    Vec3(0,0,-1),
    0.5,
    material,
    Vec3(1,0,-1)
)
```

Motion:

$$
C(t)=(0,0,-1)+t(1,0,0)
$$

Now your camera + ray + sphere together fully implement motion blur correctly.


In [6]:
import sys 
sys.path.append("../../../Ray Tracing next Week")

In [7]:
import math

from util.week1.vec3 import dot
from util.week1.hittable import Hittable
from util.week1.ray import Ray


class Sphere(Hittable):

    def __init__(
        self,
        center1,
        radius,
        material,
        center2=None
    ):
        """
        ============================================
        SPHERE CONSTRUCTOR
        ============================================

        Supports:

        1. Static sphere
        2. Moving sphere

        --------------------------------------------
        STATIC:

        Sphere(center, radius, material)

        center(t) = center

        --------------------------------------------
        MOVING:

        Sphere(center1, radius, material, center2)

        center(t)=center1+t(center2-center1)

        We store motion as a ray.
        """

        self.radius = max(0.0, radius)
        self.mat = material

        # ============================================
        # STATIC SPHERE
        # OLD behavior
        # ============================================
        if center2 is None:

            # CHANGED:
            # store zero-velocity ray
            self.center = Ray(
                center1,
                center1 * 0
            )

        # ============================================
        # MOVING SPHERE
        # NEW behavior
        # ============================================
        else:

            # center(t)=center1+t(center2-center1)

            self.center = Ray(
                center1,
                center2 - center1
            )

    # ==================================================
    # HIT TEST
    # ==================================================
    def hit(self, r, ray_t, rec):

        """
        ============================================
        STEP 1:
        Compute current center at ray time
        ============================================

        Formula:

        C(t)=C1+t(C2-C1)

        CHANGED:
        Before:
            fixed center

        Now:
            moving center
        """

        current_center = self.center.at(r.time())

        """
        Sphere equation:

        (P-C(t))·(P-C(t)) = r²

        Ray equation:

        P = O+tD
        """

        # ============================================
        # Vector from ray origin to moving center
        # ============================================
        oc = current_center - r.origin()

        # a = D·D
        a = dot(
            r.direction(),
            r.direction()
        )

        # h = D·oc
        h = dot(
            r.direction(),
            oc
        )

        # c = oc·oc - r²
        c = dot(
            oc,
            oc
        ) - self.radius * self.radius

        # ============================================
        # Discriminant
        #
        # h²-ac
        # ============================================
        discriminant = h * h - a * c

        if discriminant < 0:
            return False

        sqrtd = math.sqrt(discriminant)

        # ============================================
        # Find nearest root
        # ============================================
        root = (h - sqrtd) / a

        if not ray_t.surrounds(root):

            root = (h + sqrtd) / a

            if not ray_t.surrounds(root):
                return False

        # ============================================
        # Store hit
        # ============================================
        rec.t = root

        # P=O+tD
        rec.p = r.at(root)

        """
        ============================================
        CHANGED:
        normal uses current_center

        Before:
            rec.p - self.center

        Wrong for moving sphere

        Now:
            rec.p - current_center
        ============================================
        """

        outward_normal = (
            rec.p - current_center
        ) / self.radius

        rec.set_face_normal(
            r,
            outward_normal
        )

        # material
        rec.mat = self.mat

        return True